# Train the Nimbus adapter with QLoRA on a free GPU

Companion notebook for **llm-finetune-lab** by Asad Aslam. Works on Google Colab and Kaggle.

It runs the real pipeline from the repo, so training here is identical to training on
your own machine. Nothing is copied into the notebook that could drift out of sync.

**Before you run anything, turn the GPU on:**

- Colab: Runtime > Change runtime type > T4 GPU
- Kaggle: right sidebar > Accelerator > GPU T4 x2 (and switch Internet on)

QLoRA needs a CUDA GPU. On CPU the pipeline falls back to plain LoRA and tells you so.

## 1. Get the code

In [ ]:
REPO = 'https://github.com/asadaslam556/llm-finetune-lab.git'  # change to your fork if needed

!git clone -q {REPO} lab
%cd lab
!pip install -q -e ".[train,quant]"

## 2. Optional: Hugging Face token

Only needed for gated models like Llama or Gemma. The default Qwen model is public.
On Colab, add a secret called `HF_TOKEN` with the key icon in the left sidebar.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF token loaded from Colab secrets')
except Exception:
    print('no HF token, fine for public models')

## 3. Pick a model and check the plan

The plan shows whether this GPU will run true 4-bit QLoRA, and the estimated VRAM.
A T4 has no bf16, so expect `compute_dtype: float16`, which is fine.

The default here is `Qwen/Qwen2.5-1.5B-Instruct` (Apache-2.0). `Qwen/Qwen2.5-7B-Instruct` also fits a T4 in 4-bit, but merging it later needs about 15 GB of RAM on your own machine. Avoid Qwen2.5-3B: its licence is non-commercial.

In [ ]:
os.environ['LFL_BASE_MODEL_HF'] = 'Qwen/Qwen2.5-1.5B-Instruct'  # Apache-2.0; avoid 3B, it is research-licensed
os.environ['LFL_NUM_EPOCHS'] = '3'   # tiny dataset, a few passes help; try '6' if answers mix up facts

!finetune-lab plan

## 4. Train

Ingest, prepare, pull, profile, fine-tune and evaluate. Export happens on your machine, where Ollama lives.

In [ ]:
!finetune-lab run --real --stages ingest prepare pull_base profile finetune evaluate

## 5. Look at the results

In [ ]:
import json, glob
run = sorted(glob.glob('artifacts/2*'))[-1]
info = json.load(open(f'{run}/adapter/ADAPTER_INFO.json'))
print('strategy:', info['config']['strategy'], '| final loss:', info['final_loss'],
      '| trainable:', info.get('trainable_pct'), '%')
report = json.load(open(f'{run}/eval_report.json'))
print('overlap F1:', report['overlap_f1'], '| keyword hit rate:', report['keyword_hit_rate'])
for ex in report['examples'][:3]:
    print('
Q:', ex['question'], '
A:', ex['prediction'])

## 6. Download the adapter

**Do this before you close the tab.** Notebook disks are wiped when the session ends. The adapter is about 70 MB, not the whole model. If the automatic download spins forever, stop the cell and use the file browser as the cell prints.

In [ ]:
import shutil
zip_path = shutil.make_archive('nimbus-adapter', 'zip', f'{run}/adapter')
print('Saved', zip_path)
print('Most reliable way to download it: left sidebar > folder icon > lab > nimbus-adapter.zip > three dots > Download.')
try:
    from google.colab import files
    files.download(zip_path)  # can hang in some browsers; the file browser always works
except Exception:
    pass

## 7. Back on your machine

1. Do one dry run so `artifacts/<run-id>/` exists: `finetune-lab run`
2. Unzip `nimbus-adapter.zip` into `artifacts/<run-id>/adapter/`
3. Export and deploy for real:

```bash
finetune-lab run --real --stages export_deploy
ollama run nimbus-support
```

The export stage merges the adapter into a full-precision base before converting to GGUF.
Merging into the 4-bit base would lose the adapter, see docs/qlora.md.